# Notebook 03 — Security Effectiveness

Evaluates **technical security effectiveness** of biometric systems through:

**Part 1 — Overall performance:** FAR, FRR, EER, and AUC-ROC across three system quality levels (ISO/IEC 19795-1:2021 aligned).

**Part 2 — Demographic fairness:** Whether security effectiveness is equitable across groups (DPD, EOD, EqOdds, DIR per Hardt et al. 2016).

Uneven effectiveness creates both impersonation risk (FAR disparity) and denial-of-access risk (FRR disparity) — relevant to GDPR Art.32 and Equality Act 2010 s.19.


## Part 1 — Overall security performance


In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from biometric_auth.engine.metrics import run_evaluation
from biometric_auth.parsers.score_file import load_scores

DATA_DIR = Path('../data/synthetic')
systems = {
    'High accuracy':   load_scores(DATA_DIR / 'scores_high_accuracy.csv'),
    'Medium accuracy': load_scores(DATA_DIR / 'scores_medium_accuracy.csv'),
    'Low accuracy':    load_scores(DATA_DIR / 'scores_low_accuracy.csv'),
}
eval_results = {name: run_evaluation(df, algorithm_name=name) for name, df in systems.items()}


In [ ]:
sec_rows = []
for name, r in eval_results.items():
    sec_rows.append({
        'System': name,
        'FAR': f'{r.far:.3%}',
        'FRR': f'{r.frr:.3%}',
        'EER': f'{r.eer:.3%}',
        'AUC-ROC': f'{r.auc_roc:.5f}',
        'Verdict': 'Effective' if r.eer < 0.03 else ('Marginal' if r.eer < 0.10 else 'Ineffective'),
    })
pd.DataFrame(sec_rows)


In [ ]:
# DET-style view: FAR vs FRR across systems at EER point
colours = ['#003087', '#007A73', '#C8102E']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

eers = [r.eer * 100 for r in eval_results.values()]
fars = [r.far * 100 for r in eval_results.values()]
frrs = [r.frr * 100 for r in eval_results.values()]
labels = list(eval_results.keys())

axes[0].bar(labels, eers, color=colours, alpha=0.85)
axes[0].set_ylabel('EER (%)'); axes[0].set_title('Equal Error Rate')
axes[1].bar(labels, fars, color='#C8102E', alpha=0.7, label='FAR')
axes[1].bar(labels, frrs, color='#003087', alpha=0.5, label='FRR')
axes[1].set_ylabel('Rate (%)'); axes[1].set_title('FAR and FRR at EER threshold')
axes[1].legend()
plt.tight_layout(); plt.show()


## Part 2 — Demographic fairness (equitable security effectiveness)

NIST FRVT 2019 showed commercial systems can perform unevenly across demographic groups. We test whether security metrics (FAR/FRR/EER) are consistent across groups.


In [ ]:
from biometric_auth.engine.bias import run_bias_analysis
from biometric_auth.data.synthetic import generate_synthetic_multigroup

biased = generate_synthetic_multigroup(
    groups=['group_A', 'group_B', 'group_C'],
    eer_by_group={'group_A': 0.02, 'group_B': 0.15, 'group_C': 0.08},
    seed=42,
)
bias_result = run_bias_analysis(biased, algorithm_name='biased_system')
print(f'Verdict: {bias_result.fairness_verdict}')
print(f'EqOdds:  {bias_result.equalised_odds_difference:.4f}')
print(f'DIR:     {bias_result.disparate_impact_ratio:.4f}')


In [ ]:
rows = []
for g in bias_result.group_results.values():
    rows.append({'Group': g.group_name, 'FAR': f'{g.far:.3%}',
                 'FRR': f'{g.frr:.3%}', 'EER': f'{g.eer:.3%}',
                 'AUC-ROC': f'{g.auc_roc:.4f}'})
pd.DataFrame(rows)


In [ ]:
groups = sorted(bias_result.group_results.keys())
fars = [bias_result.group_results[g].far * 100 for g in groups]
frrs = [bias_result.group_results[g].frr * 100 for g in groups]

x = np.arange(len(groups)); w = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - w/2, fars, w, label='FAR (%)', color='#C8102E', alpha=0.8)
ax.bar(x + w/2, frrs, w, label='FRR (%)', color='#003087', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(groups)
ax.set_ylabel('Rate (%)')
ax.set_title(f'Security Effectiveness by Group\nFairness verdict: {bias_result.fairness_verdict}')
ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
fair = generate_synthetic_multigroup(
    groups=['group_A', 'group_B', 'group_C'],
    eer_by_group={'group_A': 0.05, 'group_B': 0.052, 'group_C': 0.051},
    seed=99,
)
fair_result = run_bias_analysis(fair, algorithm_name='fair_system')
print(f'Balanced system verdict: {fair_result.fairness_verdict}')
print(f'  EqOdds: {fair_result.equalised_odds_difference:.4f}')
print(f'  DIR:    {fair_result.disparate_impact_ratio:.4f}')
